# Tema 2 · Laboratorio — Tu primer MLP en Keras

**Aprendizaje Profundo · CUNEF Universidad**

En este laboratorio construimos y entrenamos un **perceptrón multicapa (MLP)** con Keras para clasificar el dataset **Wine**: 13 propiedades químicas de un vino → uno de 3 cultivares.

Es la versión en código de lo que viste en la teoría: capas `Dense`, activaciones y **backpropagation** con `model.fit`.

> Ejecuta las celdas en orden. En Colab no necesitas instalar nada.

## 1 · Los datos: Wine

178 vinos, 13 variables numéricas (alcohol, acidez, fenoles…) y 3 clases. Un problema pequeño y limpio, ideal para un primer MLP.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

data = load_wine()
X, y = data.data, data.target
print('X:', X.shape, ' clases:', np.unique(y), ' nombres:', list(data.target_names))

## 2 · Preprocesado: estandarizar y partir

Las 13 variables están en escalas muy distintas. Las **estandarizamos** (media 0, desviación 1) — un preprocesado clave para que el entrenamiento funcione bien. Y separamos train/test.

> Importante: el `scaler` se ajusta **solo con el train** y luego se aplica al test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # ajusta y transforma con el train
X_test = scaler.transform(X_test)          # solo transforma el test
print('train:', X_train.shape, ' test:', X_test.shape)

## 3 · El MLP

Una red sencilla: 13 entradas → capa oculta de 16 con `relu` → capa oculta de 8 con `relu` → salida de 3 con `softmax` (una probabilidad por clase). Eso es un perceptrón multicapa.

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(13,)),
    layers.Dense(16, activation='relu'),
    layers.Dense(8, activation='relu'),
    layers.Dense(3, activation='softmax'),
])
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
model.summary()

## 4 · Entrenar (backpropagation en acción)

`model.fit` hace, en cada época, el ciclo del Tema 2: propagación hacia delante, cálculo del error y **backpropagation** para ajustar los pesos.

In [ ]:
history = model.fit(X_train, y_train,
                    validation_split=0.15,
                    epochs=60, batch_size=16, verbose=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'], label='train')
ax1.plot(history.history['val_loss'], label='val')
ax1.set_title('Pérdida'); ax1.set_xlabel('época'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(history.history['accuracy'], label='train')
ax2.plot(history.history['val_accuracy'], label='val')
ax2.set_title('Accuracy'); ax2.set_xlabel('época'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5 · Evaluar en test

La medida honesta: qué tal clasifica vinos que no vio durante el entrenamiento.

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Accuracy en TEST: {test_acc:.4f}')

## 6 · Comparación: ¿hace falta la capa oculta?

El Tema 2 decía que la potencia del MLP viene de las **capas ocultas**. Comparemos con un modelo **sin** capa oculta (equivale a un perceptrón / regresión logística).

In [ ]:
def entrena_y_evalua(hidden):
    caps = [keras.layers.Input(shape=(13,))]
    for h in hidden:
        caps.append(layers.Dense(h, activation='relu'))
    caps.append(layers.Dense(3, activation='softmax'))
    m = keras.Sequential(caps)
    m.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    m.fit(X_train, y_train, epochs=60, batch_size=16, verbose=0)
    return m.evaluate(X_test, y_test, verbose=0)[1]

acc_sin = entrena_y_evalua([])          # sin capa oculta
acc_con = entrena_y_evalua([16, 8])     # con dos capas ocultas
print(f'Sin capa oculta (perceptrón): {acc_sin:.4f}')
print(f'MLP con dos capas ocultas   : {acc_con:.4f}')

## 7 · Tus retos

1. **Arquitectura.** Cambia el tamaño de las capas ocultas (p. ej. `[32, 16]` o una sola `[8]`). ¿Cómo afecta a la accuracy de test?
2. **Sin estandarizar.** Repite el entrenamiento **sin** el `StandardScaler`. ¿Empeora? ¿Por qué el preprocesado importa tanto?
3. **Otro dataset.** En la carpeta *General* del sitio tienes `MLP_Heart.ipynb`. Adapta este MLP al problema de corazón (clasificación binaria: usa `sigmoid` en la salida y `binary_crossentropy`).

Cuando termines, vuelve a la [práctica interactiva](../../practica-t2.html) y relaciona las capas que has escrito aquí con las regiones que dibujabas allí.